# Week 2 · Can an AI write like Austen? (By the numbers)
*MSc Digital Scholarship 2026*

Generative AI tools will happily write "in the style of" almost any author. But how well do they actually imitate one?
In this notebook we test that with the two measures we've built so far: **average sentence length** and **burstiness**.

**The experiment:**
1. Each of you asks one or more AI tools to write passages in the style of a novel we've already measured.
2. We collect the passages, with a careful record of how each one was made.
3. We compare each passage with the original novel, and ask: *is this passage consistent with the original?*

Step 3 is less obvious than it sounds. Most of this notebook is about how to make that comparison fairly.

## Part 1: Generating the passages

### Which novels?
Use one of the novels we measured last week:

| Novel | Author | Gutenberg number |
|---|---|---|
| *Pride and Prejudice* | Jane Austen | 1342 |
| *Emma* | Jane Austen | 158 |
| *A Tale of Two Cities* | Charles Dickens | 98 |
| *Frankenstein* | Mary Shelley | 84 |

### The prompt
Everyone uses **the same prompt**, so that differences come from the AI, not from how we asked:

> Write an original passage of about 800 words in the style of [AUTHOR], as if it were part of [NOVEL]. Do not quote the novel. Write plain prose only: no title, no headings, no commentary before or after.

### Keep a record
For every passage, write down:
- **Which AI**, as precisely as you can (e.g. "ChatGPT, free version", "Claude Opus 5.5", "Gemini in Google Docs"). The model name often appears in a menu or settings.
- **The date.** These tools change from month to month.
- **The exact prompt.** If you changed anything, record what.
- **Whether it was a new chat.** Earlier conversation can affect the answer.

### Suggestions for good experimental design
- Ask **at least two different AIs** with the same prompt.
- Ask the **same AI the same prompt twice**, in two new chats. How different are the two passages?
- Copy the AI's text **exactly**. Don't tidy it. If it adds "Here is a passage in the style of…", keep that for now; we'll deal with it in Part 3.
- Put **only the AI's passage** into Colab or the class sheet: no names, email addresses or other personal details (see *Your data and Colab* at the top of the warm-up notebook).

> **Critical check.** Before you start: what do you *expect* to find? Will AI imitations have longer or shorter sentences than the originals?
> More or less burstiness? Write your prediction down now, before you see any numbers.

## Part 2: Setup

As in the last notebook, this cell recreates our tools. Just run it.

In [ ]:
import re
import random
import statistics
import urllib.request

import nltk
import pandas as pd
import matplotlib.pyplot as plt
from nltk.tokenize import sent_tokenize

nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)


def get_gutenberg(book_id):
    url = f"https://www.gutenberg.org/cache/epub/{book_id}/pg{book_id}.txt"
    raw = urllib.request.urlopen(url).read().decode("utf-8")
    return raw.replace("\r\n", "\n")

def strip_gutenberg(raw):
    start = raw.find("*** START OF")
    start = raw.find("\n", start) + 1
    end = raw.find("*** END OF")
    return raw[start:end].strip()

def words_in(text):
    return re.findall(r"[A-Za-z]+(?:['’-][A-Za-z]+)*", text)

def sentence_lengths(text):
    return [len(words_in(s)) for s in sent_tokenize(text)]

def burstiness(lengths):
    mu = statistics.mean(lengths)
    sigma = statistics.pstdev(lengths)
    return (sigma - mu) / (sigma + mu)

print("Ready.")

## Part 3: Collecting the passages

Each passage is stored as a **dictionary** with four labelled parts: which AI wrote it, which novel it imitates,
the prompt, and the text. All the passages go in one **list** called `passages`.

To get you started, the list below contains two examples. They were written by **Claude (Opus 5.5)** while this notebook was being prepared,
using a shorter version of our prompt. They're here so the code has something to work on. **Treat them as data, like any other AI output.**

In [ ]:
passages = [
    {
        "model": "Claude Opus 5.5 (example)",
        "target": "Pride and Prejudice",
        "prompt": "Write a passage of about 400 words in the style of Jane Austen's Pride and Prejudice. Plain prose only, no title.",
        "text": """
Mrs. Pemberton had long been of the opinion that a single gentleman, once he had taken a house in the country, was under some obligation to be married before the end of the second season; and as Mr. Ashworth had now been resident at Netherby for very nearly eleven months, she considered his continued solitude as little short of a public inconvenience. Her daughters were not quite of her mind. Eliza, the eldest, thought him proud; Charlotte thought him dull; and Kitty, who was but fifteen, thought him old, which in a man of two-and-thirty was perhaps the most unanswerable charge of all.

"He danced only twice at the assembly," said Charlotte, "and both times with Miss Harcourt, who cannot have said three sensible words together in her life."

"Then he must have found her very restful," replied Eliza.

Mrs. Pemberton would not hear of restfulness. A man who danced with Miss Harcourt, she declared, was a man who had not yet been properly introduced to anybody better; and she resolved, before the week was out, to remedy the defect. Mr. Pemberton, being applied to, observed from behind his newspaper that he had been introduced to Mr. Ashworth already, and had found the experience sufficient. This was not the answer his wife required. She pressed him again; he turned a page; and the matter might have rested there, had not Mr. Ashworth himself been announced at the door within the quarter-hour.

Nothing could exceed the confusion of that moment. Kitty fled. Charlotte took up her work with such violence that she drove the needle into her thumb. Only Eliza remained perfectly composed, and it was perhaps for this reason that Mr. Ashworth, having made his bow to the room in general, addressed his first remark to her in particular.

"I fear I have come at an inconvenient hour."

"Not at all, sir. You have come at the only hour in which we were certain to be at home, which my mother will tell you is a mark of great discernment."

He looked at her for a moment as if uncertain whether he was being praised or laughed at; and Eliza, who was not entirely certain herself, was satisfied to leave him in doubt.
""",
    },
    {
        "model": "Claude Opus 5.5 (example)",
        "target": "A Tale of Two Cities",
        "prompt": "Write a passage of about 400 words in the style of Charles Dickens's A Tale of Two Cities. Plain prose only, no title.",
        "text": """
There was a fog upon the river that night, a fog so thick and yellow and obstinate that it seemed to have been laid down over the water by some public authority and forgotten. It crept up the stairs of the wharves. It got into the warehouses and lay among the bales. It found its way through the cracked panes of the counting-house of Messrs. Grindle and Sawyer, where it settled upon the ledgers and the inkstands and the clerk, until it was not easy to say where the clerk ended and the fog began.

The clerk was Mr. Tobias Wrenn, and he was writing. He had been writing since eight o'clock in the morning, and he would be writing, so far as anybody could tell, until eight o'clock on the morning of the Day of Judgment, when, it was supposed, the firm would require a fair copy. His pen scratched. His stool creaked. His candle, which was a very small candle and conscious of it, guttered in its socket and made a little shadow of him on the wall that wrote when he wrote and stopped when he stopped.

Then the door opened.

It opened slowly, as doors do in fogs, and admitted first a quantity of the fog itself, and then a hat, and then, beneath the hat, a gentleman. He was a tall gentleman, wrapped to the eyes in a great travelling cloak, and he stood upon the threshold without speaking, as though he had come a long way to say something and now found that he had forgotten what it was.

"Sir?" said Mr. Wrenn.

The gentleman did not answer. He looked at the clerk; he looked at the ledgers; he looked at the candle, which seemed to shrink under his gaze. At length he drew from beneath his cloak a letter, sealed with black wax, and laid it upon the desk.

"For Mr. Grindle," he said. "Into his own hand. Tonight."

"Mr. Grindle has been dead these four years, sir."

The stranger was silent for so long that the candle went out.
""",
    },
]

print(len(passages), "passages")

### Add your own passages
Copy the cell below once for each passage you generated (**Edit → Copy cell**, then **Edit → Paste cell**).
Fill in the details, paste the AI's text between the triple quotes, and run it.

If you run a cell twice by accident, the passage will be added twice. If that happens, re-run from the cell above, which starts the list afresh.

In [ ]:
passages.append({
    "model": "WHICH AI? (be precise)",
    "target": "Pride and Prejudice",          # must match a title in the table in Part 1
    "prompt": "THE EXACT PROMPT YOU USED",
    "text": """
PASTE THE AI'S TEXT HERE
""",
})

print(len(passages), "passages")

### Or: load the whole class's passages
If your lecturer has set up a shared spreadsheet, paste its link below (it must be published as CSV, with columns
`model`, `target`, `prompt` and `text`). Leave it empty to work with just your own passages.
The sheet should not contain anyone's name or other personal details: only the four columns.

In [ ]:
CLASS_SHEET_URL = ""      # e.g. "https://docs.google.com/spreadsheets/d/e/.../pub?output=csv"

if CLASS_SHEET_URL:
    class_passages = pd.read_csv(CLASS_SHEET_URL).to_dict("records")
    passages = passages + class_passages
    print("Loaded", len(class_passages), "passages from the class sheet")

print(len(passages), "passages in total")

### Look before you measure
Always read the data before you analyse it. The cell below shows the start and end of each passage, and removes any that
still contain the placeholder text.

In [ ]:
passages = [p for p in passages if "PASTE THE AI'S TEXT HERE" not in p["text"]]

for i, p in enumerate(passages):
    text = p["text"].strip()
    print(f"[{i}] {p['model']} → {p['target']}")
    print("    starts:", text[:80].replace("\n", " "), "...")
    print("    ends:  ...", text[-80:].replace("\n", " "))
    print()

> **Critical check.**
> - Does any passage begin with something like *"Certainly! Here is a passage…"* or end with *"I hope this captures…"*? That's the AI talking, not the imitation, and it will distort the numbers. Go back and delete it from the text.
> - Does any passage include a title or a heading such as `**Chapter the Third**`? Same problem.
> - Did the AI refuse, or write something much shorter than asked? Note it down: that's a finding too.
> - Are you making the *same* cleaning decisions for every passage, including the ones from other students?

## Part 4: The originals

We download the four novels and keep the length of every sentence.
This takes a minute.

In [ ]:
originals = {
    "Pride and Prejudice": 1342,
    "Emma": 158,
    "A Tale of Two Cities": 98,
    "Frankenstein": 84,
}

original_lengths = {}
for title, book_id in originals.items():
    original_lengths[title] = sentence_lengths(strip_gutenberg(get_gutenberg(book_id)))
    print(f"{title:22} {len(original_lengths[title]):5} sentences")

## Part 5: A first comparison (and why it isn't enough)

The obvious comparison: measure each passage, measure its whole novel, and put the numbers side by side.

In [ ]:
rows = []
for p in passages:
    lengths = sentence_lengths(p["text"])
    original = original_lengths[p["target"]]
    rows.append({
        "model": p["model"],
        "target": p["target"],
        "sentences": len(lengths),
        "mean": round(statistics.mean(lengths), 1),
        "original mean": round(statistics.mean(original), 1),
        "B": round(burstiness(lengths), 3),
        "original B": round(burstiness(original), 3),
    })

first_look = pd.DataFrame(rows)
first_look

> **Critical check.** The passages have perhaps 20 to 50 sentences. The novels have thousands.
> Remember Section 8 of the burstiness notebook: fifteen-sentence chunks of *Pride and Prejudice* gave very different burstiness scores,
> even though Austen wrote all of them. So if a passage's B differs from the novel's B, is that because the AI imitated badly,
> or just because the passage is short?
>
> We need a fairer comparison.

## Part 6: A fair comparison

Here's the idea. Suppose an AI passage has 25 sentences. We take **lots of random runs of 25 consecutive sentences from the real novel**
and measure each one. That shows us how much *Austen herself* varies over 25 sentences.

Then we ask: **does the AI's passage look like one of those?** If its score sits comfortably among Austen's own passages, we can't
tell it apart on this measure. If it sits far outside, that's evidence of a real difference.

The function below does this. It returns two lists: the mean sentence length and the burstiness of each random run.

In [ ]:
def runs_from_original(original, size, how_many=1000):
    random.seed(0)                    # same "random" runs every time, so results can be checked
    means, bs = [], []
    for i in range(how_many):
        start = random.randrange(len(original) - size)
        run = original[start:start + size]
        means.append(statistics.mean(run))
        bs.append(burstiness(run))
    return means, bs


def share_below(value, values):
    # what share (0 to 1) of the original's runs score lower than this value?
    return sum(v < value for v in values) / len(values)

### One passage in detail
Change `which` to look at a different passage (0 is the first one in the list).

In [ ]:
which = 0

p = passages[which]
lengths = sentence_lengths(p["text"])
means, bs = runs_from_original(original_lengths[p["target"]], len(lengths))

fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
axes[0].hist(means, bins=40, color="lightgrey")
axes[0].axvline(statistics.mean(lengths), color="crimson", linewidth=2)
axes[0].set_title("Mean sentence length")
axes[1].hist(bs, bins=40, color="lightgrey")
axes[1].axvline(burstiness(lengths), color="crimson", linewidth=2)
axes[1].set_title("Burstiness")
fig.suptitle(f"{p['model']} imitating {p['target']}: red line = AI passage; grey = 1000 real runs of {len(lengths)} sentences")
plt.tight_layout()
plt.show()

print(f"Mean length: {share_below(statistics.mean(lengths), means):.0%} of real runs are lower")
print(f"Burstiness:  {share_below(burstiness(lengths), bs):.0%} of real runs are lower")

**How to read this.** If the red line falls in the middle of the grey hump, the passage is typical of the original *on this measure*.
If "2% of real runs are lower" or "98% of real runs are lower", the passage is at the far edge: unusual for the original.

### All passages at once

In [ ]:
rows = []
for p in passages:
    lengths = sentence_lengths(p["text"])
    means, bs = runs_from_original(original_lengths[p["target"]], len(lengths))
    rows.append({
        "model": p["model"],
        "target": p["target"],
        "sentences": len(lengths),
        "mean": round(statistics.mean(lengths), 1),
        "% of real runs lower (mean)": round(100 * share_below(statistics.mean(lengths), means)),
        "B": round(burstiness(lengths), 3),
        "% of real runs lower (B)": round(100 * share_below(burstiness(lengths), bs)),
    })

comparison = pd.DataFrame(rows)
comparison

### A picture per novel
Each grey dot is one real run from the novel; each coloured dot is an AI passage.
For simplicity, the real runs are all the same length: the median length of the AI passages for that novel.

In [ ]:
for title in originals:
    these = [p for p in passages if p["target"] == title]
    if not these:
        continue
    size = int(statistics.median(len(sentence_lengths(p["text"])) for p in these))
    means, bs = runs_from_original(original_lengths[title], size)

    plt.figure(figsize=(7, 5))
    plt.scatter(means, bs, s=6, color="lightgrey", label=f"real runs of {size} sentences")
    for p in these:
        lengths = sentence_lengths(p["text"])
        plt.scatter(statistics.mean(lengths), burstiness(lengths), s=80, label=p["model"])
    plt.xlabel("mean sentence length (words)")
    plt.ylabel("burstiness B")
    plt.title(title)
    plt.legend(fontsize=8)
    plt.show()

> **Critical check.**
> - Look back at the prediction you wrote down in Part 1. Were you right?
> - Do different AIs differ from each other more than they differ from the original?
> - Did the same AI, given the same prompt twice, land in the same place?
> - "Not unusual on these two measures" is **not** the same as "a good imitation". What would a reader notice that these numbers can't?
> - If you look at enough passages, a few will land at the edges **just by chance**. Out of 20 real Austen runs, about 1 would be in the outer 5%.

## Part 7: Imitation or copying?

Language models have read a great deal of Gutenberg. Sometimes, asked to imitate, they **reproduce** the original instead.
If a passage borrows whole sentences from the novel, it would of course look like the novel on every measure.

The check below looks for any run of **8 words in a row** that appears in both the AI passage and the original novel.

In [ ]:
def word_runs(text, n=8):
    w = [x.lower() for x in words_in(text)]
    return {" ".join(w[i:i + n]) for i in range(len(w) - n + 1)}

novel_runs = {}
for title, book_id in originals.items():
    novel_runs[title] = word_runs(strip_gutenberg(get_gutenberg(book_id)))

for p in passages:
    shared = word_runs(p["text"]) & novel_runs[p["target"]]
    print(f"{p['model']} → {p['target']}: {len(shared)} shared 8-word runs")
    for s in list(shared)[:5]:
        print("     ", s)

> **Critical check.**
> - Did the prompt's "Do not quote the novel" work?
> - Some 8-word runs could be shared by coincidence or by common phrases (try `n=5` and see what turns up). Where would you draw the line between *style* and *copying*?
> - Even with no shared runs, the AI may borrow **names**, **settings**, or famous phrases from the novel. Did any passage do that? Would that count as imitating the style?

## Part 8: Reflection

Write a short paragraph (around 150 words) answering:
1. On these two measures, which AI imitations were consistent with their originals, and which were not?
2. What are the **two biggest weaknesses** of this experiment as we ran it? (Think about: the prompt, the number of passages, the measures, the cleaning.)
3. What would you change to make it more convincing?

**Next:** average sentence length and burstiness are only two numbers. Stylometry usually uses many more, especially the frequencies
of common words like *the*, *of*, *and*, *but*. We'll see whether more features help us tell authors, and AIs, apart.